In [ ]:
# === ATLAS OWNED BRAIN on free Colab GPU (T4) ===
# Ornith-1.0-9B + your trained adapter (distilled from GLM-5.2 + Kimi K2.7).
# Setup: 1) Runtime > Change runtime type > T4 GPU.  2) Click the 🔑 (Secrets) panel,
#        add GITHUB_TOKEN and BRAIN_SECRET (toggle "Notebook access" on for both).
#        3) Runtime > Run all.  It serves your brain and auto-connects to Atlas;
#        Atlas falls back to GLM/Kimi the moment this session ends.
import json, os, re, subprocess, sys, tarfile, threading, time, urllib.request
from google.colab import userdata
GH = userdata.get('GITHUB_TOKEN'); BRAIN_SECRET = userdata.get('BRAIN_SECRET')
REGISTER_URL = "https://atlas.supremeapis.com/api/brain/register"
subprocess.run([sys.executable,"-m","pip","install","-q","transformers>=5.8.1",
  "peft","accelerate","bitsandbytes","einops","sentencepiece","flask"], check=True)
subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/"
  "download/cloudflared-linux-amd64 -O /tmp/cf && chmod +x /tmp/cf", shell=True, check=True)
# pull your private adapter
REPO,TAG="jrennie99-glitch/atlas","ornith-adapter-v1"
rel=json.load(urllib.request.urlopen(urllib.request.Request(
  f"https://api.github.com/repos/{REPO}/releases/tags/{TAG}",
  headers={"Authorization":f"Bearer {GH}","User-Agent":"colab"})))
a=next(x for x in rel["assets"] if x["name"]=="ornith_adapter.tar.gz")
open("/tmp/ad.tgz","wb").write(urllib.request.urlopen(urllib.request.Request(a["url"],
  headers={"Authorization":f"Bearer {GH}","Accept":"application/octet-stream","User-Agent":"colab"})).read())
os.makedirs("/tmp/adapter",exist_ok=True); tarfile.open("/tmp/ad.tgz").extractall("/tmp/adapter")
print("adapter ready")
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
BASE="deepreinforce-ai/Ornith-1.0-9B"
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",
  bnb_4bit_compute_dtype=torch.bfloat16,bnb_4bit_use_double_quant=True)
try:
  base=AutoModelForCausalLM.from_pretrained(BASE,quantization_config=bnb,
    device_map="auto",torch_dtype=torch.bfloat16,trust_remote_code=True)
except Exception as e:
  print("fallback:",e); from transformers import AutoModel
  base=AutoModel.from_pretrained(BASE,quantization_config=bnb,device_map="auto",
    torch_dtype=torch.bfloat16,trust_remote_code=True)
model=PeftModel.from_pretrained(base,"/tmp/adapter").eval()
tok=AutoTokenizer.from_pretrained("/tmp/adapter",trust_remote_code=True)
if tok.pad_token is None: tok.pad_token=tok.eos_token
print("model ready")
from flask import Flask, request, jsonify
app=Flask(__name__)
@app.post("/chat")
def chat():
  d=request.get_json(force=True)
  if d.get("secret")!=BRAIN_SECRET: return jsonify({"error":"bad secret"}),401
  p=tok.apply_chat_template(d.get("messages",[]),tokenize=False,add_generation_prompt=True)
  ins=tok(p,return_tensors="pt").to(model.device)
  with torch.no_grad():
    o=model.generate(**ins,max_new_tokens=int(d.get("max_tokens",512)),do_sample=True,
      temperature=float(d.get("temperature",0.7)),top_p=0.95,pad_token_id=tok.pad_token_id)
  return jsonify({"choices":[{"message":{"role":"assistant","content":
    tok.decode(o[0][ins["input_ids"].shape[1]:],skip_special_tokens=True).strip()}}]})
@app.get("/health")
def h(): return jsonify({"ok":True})
threading.Thread(target=lambda:app.run(host="0.0.0.0",port=8000),daemon=True).start()
time.sleep(4)
proc=subprocess.Popen(["/tmp/cf","tunnel","--url","http://localhost:8000","--no-autoupdate"],
  stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
url=None
for line in proc.stdout:
  m=re.search(r"https://[-a-z0-9]+\.trycloudflare\.com",line)
  if m: url=m.group(0); break
print("ATLAS BRAIN LIVE:",url)
def reg():
  try:
    r=urllib.request.Request(REGISTER_URL,data=json.dumps({"url":url,"secret":BRAIN_SECRET}).encode(),
      headers={"Content-Type":"application/json"},method="POST")
    print("registered:",urllib.request.urlopen(r,timeout=20).read().decode()[:120])
  except Exception as e: print("register err:",e)
reg()
print("Keep this tab running. Atlas auto-falls-back to GLM/Kimi when it stops.")
while True: time.sleep(150); reg()
